# 🏎️ JetBot 道路跟隨 + 交通路牌辨識 合併控制系統 (Final Project)

本專案合併了 **Project 5 (AI 道路跟隨 - ResNet-18)** 與 **Project 6 (AI 路牌辨識 - YOLOv4-tiny)**，使 JetBot 能夠在即時跟隨賽道的同時，辨識路旁的交通號誌並執行對應動作。

## 🚥 號誌控制邏輯 (採用 `yolo26` 類別順序):
- **Class 0: 道路封閉 (blocked)** ➡ 立即停下，且不再繼續行駛。
- **Class 1: 當心行人 (pedestrian)** ➡ 減速行駛 (速度調整為正常速的 0.7 倍)。
- **Class 2: 鐵路平交道 (rail)** ➡ 原地停止 **5 秒** 後繼續行駛。
- **Class 3: 停車再開 (stop)** ➡ 原地停止 **2 秒** 後繼續行駛。
- **忽略假路牌/遠處噪聲** ➡ 透過設定偵測框寬度門檻 `ALERT_WIDTH` 來決定何時觸發動作。

### 1. 載入必要套件與函式庫

In [ ]:
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

# 載入 YOLOv4-tiny TensorRT 優化模組
import pycuda.autoinit
from utils.yolo import TRT_YOLO

print("套件匯入成功 ✓")

### 2. 載入兩個 TensorRT 加速模型

In [ ]:
# 定義模型路徑
ROAD_MODEL_PATH = 'Project5_road_detect/best_steering_model_xy_trt.pth'
YOLO_MODEL_NAME = 'yolov4-tiny-416'  # 會自動讀取 yolov4-tiny-416.engine 檔案

# A. 載入道路跟隨模型 (ResNet-18 TRT)
device = torch.device('cuda')
model_road = TRTModule()
model_road.load_state_dict(torch.load(ROAD_MODEL_PATH))
print("道路跟隨 (ResNet-18 TRT) 模型載入完畢 ✓")

# B. 載入路牌辨識模型 (YOLOv4-tiny TRT)
model_sign = TRT_YOLO(YOLO_MODEL_NAME, (416, 416), 4)
print("路牌辨識 (YOLOv4-tiny TRT) 模型載入完畢 ✓")

### 3. 初始化相機與 Robot

In [ ]:
# 因為 YOLOv4-tiny 需要 416x416 輸入以精準捕捉遠處路牌
# 我們將相機解析度設為 416x416，道路跟隨在推論前再下採樣至 224x224
camera = Camera.instance(width=416, height=416, fps=10)
robot = Robot()

print("相機初始化 (416x416) ✓")
print("Robot 馬達介面啟動 ✓")

### 4. 道路跟隨影像前處理定義

In [ ]:
# ImageNet 的正規化常數 (轉為 FP16 半精度加速)
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road(image):
    """
    將 416x416 BGR 影像轉換為 ResNet-18 專用的 224x224 正規化 Tensor
    """
    # 1. 縮放為 224x224
    image = cv2.resize(image, (224, 224))
    # 2. 轉為 RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # 3. HWC -> CHW
    image = image.transpose((2, 0, 1))
    # 4. 轉為 PyTorch Tensor 並移至 GPU
    image = torch.from_numpy(image).float().to(device).half()
    # 5. 歸一化 [0, 255] -> [0, 1]
    image = image / 255.0
    # 6. 正規化 (減均值除標準差)
    image = (image - mean[:, None, None]) / std[:, None, None]
    # 7. 增加 Batch 維度
    return image.unsqueeze(0)

print("道路影像前處理定義完成 ✓")

### 5. 建立互動式 UI 滑桿與顯示畫面

In [ ]:
# 基礎前進速度滑桿
speed_gain_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Speed Gain')
# PD 控制器參數
p_gain_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.08, description='P Gain')
d_gain_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.005, value=0.00, description='D Gain')
bias_slider = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Motor Bias')

# 號誌辨識有效距離（邊框寬度門檻）
alert_width_slider = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')

# 即時串流顯示
image_widget = widgets.Image(format='jpeg', width=416, height=416)

display(
    image_widget,
    speed_gain_slider,
    p_gain_slider,
    d_gain_slider,
    bias_slider,
    alert_width_slider
)

### 6. 定義合併控制迴圈

In [ ]:
# 全域變數定義
angle_last = 0.0
stop_cooldown_until = 0.0
rail_cooldown_until = 0.0
current_action = "Following Lane"

def execute(change):
    global angle_last, stop_cooldown_until, rail_cooldown_until, current_action
    
    # 1. 取得最新相機畫面
    img = change['new']
    current_time = time.time()
    
    # --- 步驟 A: 物件偵測 (YOLOv4-tiny 路牌辨識) ---
    boxes, confs, clss = model_sign.detect(img, conf_th=0.6)
    
    # 整理偵測到的路牌資訊 (依照大小排序，以離車子最近/最大的路牌優先)
    detected_signs = []
    for box, conf, cls in zip(boxes, confs, clss):
        width = box[2] - box[0]
        detected_signs.append({"width": width, "class_id": cls, "box": box})
    
    detected_signs.sort(reverse=True, key=lambda x: x["width"])
    
    # 速度控制乘數
    speed_multiplier = 1.0
    current_action = "Following Lane"
    
    # 篩選掉冷卻期中或小於警示寬度的號誌
    active_sign = None
    ALERT_WIDTH = alert_width_slider.value
    
    for sign in detected_signs:
        cid = sign["class_id"]
        w = sign["width"]
        
        # 排除冷卻期內的重複動作
        if cid == 3 and current_time < stop_cooldown_until:
            continue
        if cid == 2 and current_time < rail_cooldown_until:
            continue
        
        # 只有當路牌尺寸夠大時，才視為有效號誌
        if w >= ALERT_WIDTH:
            active_sign = sign
            break
            
    # 執行號誌狀態機對應動作
    if active_sign is not None:
        cid = active_sign["class_id"]
        
        if cid == 3:  # 停車再開 (stop) ➡ 原地停止 2 秒
            current_action = "ACTION: Stop & Go (2s)"
            print("[SIGN] Detected STOP. Stopping for 2s...")
            robot.stop()
            time.sleep(2.0)
            stop_cooldown_until = time.time() + 10.0  # 設定 10 秒防重複冷卻
            
        elif cid == 2:  # 鐵路平交道 (rail) ➡ 原地停止 5 秒
            current_action = "ACTION: Railway (5s)"
            print("[SIGN] Detected RAILWAY. Stopping for 5s...")
            robot.stop()
            time.sleep(5.0)
            rail_cooldown_until = time.time() + 10.0  # 設定 10 秒防重複冷卻
            
        elif cid == 1:  # 當心行人 (pedestrian) ➡ 減速 0.7 倍
            current_action = "ACTION: Pedestrian (Slow 0.7x)"
            speed_multiplier = 0.7
            
        elif cid == 0:  # 道路封閉 (blocked) ➡ 永久停車
            current_action = "ACTION: Road Closed! STOP"
            print("[SIGN] Detected BLOCKED. Stopping permanently.")
            robot.stop()
            camera.unobserve(execute, names='value')  # 解除相機觀察以釋放系統
            return
            
    # --- 步驟 B: 道路跟隨 (ResNet-18 迴歸推論) ---
    road_tensor = preprocess_road(img)
    output = model_road(road_tensor).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[1])
    
    # 配合訓練時的 Y 軸處理修正邏輯
    y_adjusted = (0.5 - y) / 2.0
    
    # 計算目標角度
    angle = np.arctan2(x, y_adjusted)
    
    # PD 控制器
    p_gain = p_gain_slider.value
    d_gain = d_gain_slider.value
    bias = bias_slider.value
    
    pid = angle * p_gain + (angle - angle_last) * d_gain
    angle_last = angle
    
    # 修正公差得到 steering
    steering = pid + bias
    
    # 結合號誌乘數，計算馬達最終出力
    base_speed = speed_gain_slider.value * speed_multiplier
    left_motor = max(min(base_speed + steering, 1.0), 0.0)
    right_motor = max(min(base_speed - steering, 1.0), 0.0)
    
    # 輸出至馬達
    robot.left_motor.value = left_motor
    robot.right_motor.value = right_motor
    
    # --- 步驟 C: 影像繪製與即時顯示 ---
    display_img = np.copy(img)
    
    # 1. 標記道路中心預測點 (綠色實心圓)
    pixel_x = int(x * 208 + 208)
    pixel_y = int(y_adjusted * 208 + 208)
    cv2.circle(display_img, (pixel_x, pixel_y), 10, (0, 255, 0), -1)
    
    # 2. 標記號誌偵測框 (紅色矩形)
    for sign in detected_signs:
        box = sign["box"]
        cid = sign["class_id"]
        cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
        cv2.putText(display_img, f"ID:{cid} W:{sign['width']}", (box[0], box[1] - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                    
    # 3. 標記當前自走車行為狀態
    cv2.putText(display_img, current_action, (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                
    # 4. 更新影像 Widget
    image_widget.value = bgr8_to_jpeg(display_img)

print("合併控制邏輯 execute 定義完成 ✓")

### 7. 啟動自走車自動駕駛

In [ ]:
# 執行首次初始化觸發
execute({'new': camera.value})

# 綁定相機數值更新事件，開啟自動駕駛
camera.observe(execute, names='value')
print("🚗 JetBot 道路跟隨與路牌辨識系統已啟動！")

### 8. 手動安全停機

In [ ]:
# 取消相機觀察
camera.unobserve(execute, names='value')
time.sleep(0.5)  # 確保最後一幀影像處理完畢

# 停止馬達
robot.stop()

# 關閉相機連結與相機資源
camera.stop()

print("JetBot 馬達已安全停止 ✓")
print("相機硬體已釋放 ✓")